# Phase 6 - Model Export and Deployment Readiness

Notebook ini menjalankan Phase 6 dari PRD: memastikan model siap diexport sebagai `.pkl` dan dapat digunakan kembali untuk prediksi harga mobil berdasarkan 8 variabel independen yang dipilih.

Fitur input model:
- `Engine_size`
- `Horsepower`
- `Wheelbase`
- `Width`
- `Length`
- `Curb_weight`
- `Fuel_capacity`
- `Fuel_efficiency`

Output:
- `outputs/phase6/car_price_model.pkl`
- `outputs/phase6/deployment_readiness.json`
- `outputs/phase6/sample_prediction.csv`

## 1. Import Library

In [ ]:
from pathlib import Path
import json
import shutil

import joblib
import pandas as pd

## 2. Definisi Fitur Input

In [ ]:
FITUR = [
    "Engine_size",
    "Horsepower",
    "Wheelbase",
    "Width",
    "Length",
    "Curb_weight",
    "Fuel_capacity",
    "Fuel_efficiency",
]

FITUR

## 3. Load Model Hasil Training

In [ ]:
BASE_DIR = Path.cwd()
PHASE4_DIR = BASE_DIR / "outputs" / "phase4"
PHASE6_DIR = BASE_DIR / "outputs" / "phase6"
PHASE6_DIR.mkdir(parents=True, exist_ok=True)

source_model_path = PHASE4_DIR / "car_price_linear_regression_model.pkl"
model = joblib.load(source_model_path)

print("Model berhasil di-load:", type(model).__name__)
print("Jumlah fitur model:", model.n_features_in_)

## 4. Validasi Kesesuaian Fitur

In [ ]:
model_features = list(getattr(model, "feature_names_in_", FITUR))

if model_features != FITUR:
    raise ValueError(f"Fitur model tidak sesuai. Model: {model_features}, Expected: {FITUR}")

print("Fitur model sesuai dengan FITUR yang ditentukan.")

## 5. Export Model Final

In [ ]:
final_model_path = PHASE6_DIR / "car_price_model.pkl"
shutil.copy2(source_model_path, final_model_path)

reloaded_model = joblib.load(final_model_path)
print("Model final berhasil disimpan dan di-load ulang:", final_model_path)

## 6. Contoh Prediksi dengan DataFrame Baru

In [ ]:
sample_input = pd.DataFrame([
    {
        "Engine_size": 2.5,
        "Horsepower": 170,
        "Wheelbase": 107.0,
        "Width": 70.5,
        "Length": 187.8,
        "Curb_weight": 3.2,
        "Fuel_capacity": 17.2,
        "Fuel_efficiency": 26,
    }
])

sample_prediction = reloaded_model.predict(sample_input[FITUR])[0]
sample_output = sample_input.copy()
sample_output["Predicted_Price_in_thousands"] = sample_prediction
sample_output["Predicted_Price_USD"] = sample_prediction * 1000

sample_output

## 7. Simpan Deployment Readiness

In [ ]:
sample_prediction_path = PHASE6_DIR / "sample_prediction.csv"
readiness_path = PHASE6_DIR / "deployment_readiness.json"

sample_output.to_csv(sample_prediction_path, index=False)

readiness = {
    "status": "ready",
    "model_type": type(reloaded_model).__name__,
    "model_path": str(final_model_path),
    "required_features": FITUR,
    "feature_count": len(FITUR),
    "sample_prediction_in_thousands": float(sample_prediction),
    "sample_prediction_usd": float(sample_prediction * 1000),
    "sample_prediction_path": str(sample_prediction_path),
}

readiness_path.write_text(json.dumps(readiness, indent=2), encoding="utf-8")

print("Deployment readiness saved to:", readiness_path)
print(json.dumps(readiness, indent=2))

## Phase 6 Checklist

- Model berhasil diexport sebagai `car_price_model.pkl`.
- Model dapat di-load kembali tanpa error.
- Fitur input model tervalidasi sesuai 8 variabel independen.
- Contoh prediksi menggunakan dataframe berhasil dibuat.
- Model siap digunakan oleh API atau aplikasi web pada tahap deployment.